In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier
#using week 9 model

In [2]:
# Find project root from either repo root or notebooks/.
project_dir = Path.cwd().resolve()
if not (project_dir / "models").exists():
    project_dir = project_dir.parent

model_path = project_dir / "models" / "week9_best_model.pkl"
data_path = project_dir / "data" / "processed" / "matchupDiff_week5_features.csv"

with open(model_path, "rb") as f:
    week9Model = pickle.load(f)


c:\Users\Colin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.4.1.post1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Colin\AppData\Local\Temp\ipykernel_8196\196989989.py:10: UserWarning: [07:11:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\data\../common/error_msg.h:82: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and seria

In [3]:
# Pull Week 9 settings from payload.
model_family = week9Model.get("modelFamily", "xgboost")
week9_params_raw = dict(week9Model.get("params", {}))
decision_threshold = float(week9Model["threshold"])
feature_cols = list(week9Model["featureCols"])

# Convert Week 9 key names to XGBoost constructor names.
param_key_map = {
    "nEstimators": "n_estimators",
    "maxDepth": "max_depth",
    "learningRate": "learning_rate",
    "colsampleBytree": "colsample_bytree",
    "minChildWeight": "min_child_weight",
    "regLambda": "reg_lambda",
    "regAlpha": "reg_alpha",
    "randomState": "random_state",
}

In [4]:
def build_week9_pipeline():
    if model_family != "xgboost":
        raise ValueError(f"This Week 10 notebook currently supports Week 9 xgboost model only. Found: {model_family}")

    xgb_params = {}
    for k, v in week9_params_raw.items():
        if k in {"modelFamily", "scaleFeatures", "threshold"}:
            continue
        xgb_params[param_key_map.get(k, k)] = v

    # Keep stable defaults if they were not saved.
    xgb_params.setdefault("objective", "binary:logistic")
    xgb_params.setdefault("eval_metric", "logloss")
    xgb_params.setdefault("n_jobs", -1)
    xgb_params.setdefault("tree_method", "hist")

    model = XGBClassifier(**xgb_params)
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])
    
df = pd.read_csv(data_path)
df["y"] = df["y"].astype(int)

print(f"Loaded payload: {model_path}")
print(f"Model family from Week 9: {model_family}")
print(f"Threshold from Week 9: {decision_threshold}")
print(f"Week 9 feature count: {len(feature_cols)}")

Loaded payload: C:\Users\Colin\Desktop\Github\BC-AppliedAnalyticsProject\models\week9_best_model.pkl
Model family from Week 9: xgboost
Threshold from Week 9: 0.45
Week 9 feature count: 108


In [5]:
# remove any leakage possible
leak_cols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]
id_cols = ["team1Id", "team2Id"]
drop_cols = ["Season", "y"] + leak_cols + id_cols

#traintest split same as week 9 
train_df = df[df["Season"] <= 2021].copy()
val_df = df[df["Season"] == 2022].copy()
test_df = df[df["Season"].between(2023, 2025)].copy()

def build_xy(split_df, base_feature_cols):
    x = split_df.drop(columns=drop_cols, errors="ignore").select_dtypes(include=[np.number]).copy()
    x = x.reindex(columns=base_feature_cols)
    y = split_df["y"].astype(int).copy()
    return x, y

x_train_base, y_train = build_xy(train_df, feature_cols)
x_val_base, y_val = build_xy(val_df, feature_cols)
x_test_base, y_test = build_xy(test_df, feature_cols)

print(f"Train: {x_train_base.shape}, Val: {x_val_base.shape}, Test: {x_test_base.shape}")

Train: (1162, 108), Val: (63, 108), Test: (128, 108)


In [6]:
def score_split(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
    }
    cm = pd.DataFrame(
        confusion_matrix(y_true, preds),
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"],
    )
    return metrics, cm

In [7]:
def evaluate_variant(name, x_train, y_train, x_val, y_val, x_test, y_test):
    # rebuilds week9 model with same params
    pipe = build_week9_pipeline()
    pipe.fit(x_train, y_train)

    train_proba = pipe.predict_proba(x_train)[:, 1]
    val_proba = pipe.predict_proba(x_val)[:, 1]
    test_proba = pipe.predict_proba(x_test)[:, 1]

    train_m, train_cm = score_split(y_train, train_proba, decision_threshold)
    val_m, val_cm = score_split(y_val, val_proba, decision_threshold)
    test_m, test_cm = score_split(y_test, test_proba, decision_threshold)

    metrics_df = pd.DataFrame([
        {"variant": name, "dataset": "train", **train_m},
        {"variant": name, "dataset": "validation", **val_m},
        {"variant": name, "dataset": "test", **test_m},
    ])

    return {
        "variant": name,
        "pipeline": pipe,
        "metrics_df": metrics_df,
        "confusion_matrices": {
            "train": train_cm,
            "validation": val_cm,
            "test": test_cm,
        },
    }

In [8]:
def find_redundant_columns(x_train, corr_threshold=0.99995): #remove near dupes
    corr_abs = x_train.corr(numeric_only=True).abs()
    cols = corr_abs.columns.tolist()
    redundant = set()

    for i, left in enumerate(cols):
        if left in redundant:
            continue
        for j in range(i + 1, len(cols)):
            right = cols[j]
            if right in redundant:
                continue
            if pd.notna(corr_abs.iloc[i, j]) and corr_abs.iloc[i, j] >= corr_threshold:
                redundant.add(right)

    return sorted(redundant)

In [9]:
def clip_by_train_quantiles(x_train, x_val, x_test, lower_q=0.01, upper_q=0.99): #quartiles
    lower = x_train.quantile(lower_q)
    upper = x_train.quantile(upper_q)
    return (
        x_train.clip(lower=lower, upper=upper, axis=1),
        x_val.clip(lower=lower, upper=upper, axis=1),
        x_test.clip(lower=lower, upper=upper, axis=1),
    )

In [10]:
def add_simple_error_flags(x_df): #adds flags for seed diff and adjusted efficiency margin
    out = x_df.copy()

    out["absSeedDiff"] = out["Seed_diff"].abs() if "Seed_diff" in out.columns else 0.0
    out["absAdjEmDiff"] = out["AdjEM_diff"].abs() if "AdjEM_diff" in out.columns else 0.0

    # Simple context flags for hard matchups.
    out["tossupMatchupFlag"] = ((out["absSeedDiff"] <= 2) & (out["absAdjEmDiff"] <= 6)).astype(int)
    out["midAdjEmGapFlag"] = ((out["absAdjEmDiff"] > 5.5) & (out["absAdjEmDiff"] <= 15)).astype(int)

    return out

In [11]:

variant_results = {} #4 total variants: baseline and then the three improvements

# Variant 0: baseline inputs from Week 9 feature set.
variant_results["baseline"] = evaluate_variant(
    "baseline", x_train_base, y_train, x_val_base, y_val, x_test_base, y_test
)

# Variant 1: remove near duplicates 
redundant_cols = find_redundant_columns(x_train_base)
x_train_v1 = x_train_base.drop(columns=redundant_cols, errors="ignore")
x_val_v1 = x_val_base.reindex(columns=x_train_v1.columns)
x_test_v1 = x_test_base.reindex(columns=x_train_v1.columns)
variant_results["improvement1_prune"] = evaluate_variant(
    "improvement1_prune", x_train_v1, y_train, x_val_v1, y_val, x_test_v1, y_test
)

# Variant 2: reduce outliers as model may be overly dependent  
x_train_v2, x_val_v2, x_test_v2 = clip_by_train_quantiles(x_train_v1, x_val_v1, x_test_v1)
variant_results["improvement2_clip"] = evaluate_variant(
    "improvement2_clip", x_train_v2, y_train, x_val_v2, y_val, x_test_v2, y_test
)

# Variant 3: add a couple of easy matchup flags.  
x_train_v3 = add_simple_error_flags(x_train_v2)
x_val_v3 = add_simple_error_flags(x_val_v2).reindex(columns=x_train_v3.columns)
x_test_v3 = add_simple_error_flags(x_test_v2).reindex(columns=x_train_v3.columns)
variant_results["improvement3_flags"] = evaluate_variant(
    "improvement3_flags", x_train_v3, y_train, x_val_v3, y_val, x_test_v3, y_test
)

print(f"Redundant columns removed: {len(redundant_cols)}")

Redundant columns removed: 12


In [12]:
#results
all_metrics_df = pd.concat([v["metrics_df"] for v in variant_results.values()], ignore_index=True)
all_metrics_df[["accuracy", "precision", "recall"]] = all_metrics_df[["accuracy", "precision", "recall"]].round(4)

print("Metrics by variant:")
display(all_metrics_df.sort_values(["variant", "dataset"]))

val_rank = all_metrics_df[all_metrics_df["dataset"] == "validation"].copy()
val_rank = val_rank.sort_values(["accuracy", "recall"], ascending=False).reset_index(drop=True)
chosen_variant = val_rank.loc[0, "variant"]

print(f"Chosen variant: {chosen_variant}")
display(variant_results[chosen_variant]["metrics_df"].round(4))

print("Chosen variant confusion matrices:")
for split in ["train", "validation", "test"]:
    print(f"\n{split.upper()}")
    display(variant_results[chosen_variant]["confusion_matrices"][split])

Metrics by variant:


,variant,dataset,accuracy,precision,recall
2,baseline,test,0.6719,0.7009,0.8824
0,baseline,train,0.9191,0.8966,0.9949
1,baseline,validation,0.7778,0.7600,0.9500
5,improvement1_prune,test,0.6953,0.7091,0.9176
3,improvement1_prune,train,0.9182,0.8946,0.9962
4,improvement1_prune,validation,0.7460,0.7400,0.9250
8,improvement2_clip,test,0.6953,0.7091,0.9176
6,improvement2_clip,train,0.9234,0.9008,0.9962
7,improvement2_clip,validation,0.7778,0.7600,0.9500
11,improvement3_flags,test,0.7109,0.7182,0.9294


Chosen variant: baseline


,variant,dataset,accuracy,precision,recall
0,baseline,train,0.9191,0.8966,0.9949
1,baseline,validation,0.7778,0.7600,0.9500
2,baseline,test,0.6719,0.7009,0.8824


Chosen variant confusion matrices:

TRAIN


,Pred 0,Pred 1
Actual 0,288,90
Actual 1,4,780



VALIDATION


,Pred 0,Pred 1
Actual 0,11,12
Actual 1,2,38



TEST


,Pred 0,Pred 1
Actual 0,11,32
Actual 1,10,75
